# Chapter 20 — Deep Feedforward Neural Networks II: Dropout, DropConnect & Normalization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch20_deep_feedforward_neural_networks_ii/ch20_notebook.ipynb)

Companion notebook for **Chapter 20** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Dropout randomly removes nodes during training, so the network cannot rely on any single neuron.
- With L nodes, dropout implicitly trains an ensemble of up to 2^L thinned networks at no extra cost.
- At test time every node is active, but weights are scaled by the retention probability to approximate that ensemble.
- DropConnect is the same idea applied to individual weights rather than whole nodes.
- Batch normalization normalizes each neuron across the mini-batch; layer normalization normalizes each example across its features, which is why it works with a batch size of 1.

## Example 1: Dropout — Step-by-Step Manual Demonstration

We build dropout from scratch to show exactly what happens during training and testing. During training, a random binary mask is applied to the layer — zeroing out randomly chosen nodes for each mini-batch. Each mini-batch gets a different thinned network. With L=6 nodes, there are 2^6=64 possible thinned networks — we're implicitly training an ensemble of 64 models in parallel! At test time, all nodes are active but weights are multiplied by p_keep (the retention probability) to approximate the average of all 64 networks. We also show the key difference from DropConnect: Dropout zeros entire node rows; DropConnect zeros individual weight connections.

In [1]:
import numpy as np
np.random.seed(42)

L      = 6
p_keep = 0.5
layer  = np.random.randn(4, L)  # 4 examples x 6 nodes

print("=== Dropout: Training Phase ===\n")
print("Layer outputs BEFORE dropout (4 examples x 6 nodes):")
print(layer.round(4))
print()
for mb in range(1, 4):
    mask = (np.random.rand(L) > (1 - p_keep)).astype(float)
    out  = layer * mask
    dropped = [int(i) for i in np.where(mask == 0)[0]]
    print(f"Mini-batch {mb}: mask = {mask.astype(int)}  dropped nodes: {dropped}")
    print(f"  Output (example 1): {out[0].round(4)}")
    print(f"  Active nodes: {int(mask.sum())}/{L}  (thinned network)\n")

print("=== Test Phase: No Dropout, Scale Weights ===\n")
W = np.random.randn(L, 3)
print(f"At test time: all {L} nodes active BUT weights * p_keep={p_keep}")
print(f"  Original weight row 1: {W[0].round(4)}")
print(f"  Scaled   weight row 1: {(W[0]*p_keep).round(4)}")
print(f"  This approximates averaging over all 2^{L}={2**L} thinned networks\n")

print("=== Dropout vs DropConnect ===\n")
W_orig = np.array([[0.5,-0.3,0.8],[0.2,0.9,-0.4],[-0.6,0.1,0.7]])
print("Original W:"); print(W_orig)

W_drop = W_orig * np.array([1,0,1])[:,None]   # node 2 dropped (row zeroed)
print("\nDropout (node 2 dropped — full row 2 zeroed):")
print(W_drop)

dc_mask = np.array([[1,0,1],[1,1,0],[0,1,1]])
W_dc = W_orig * dc_mask
print("\nDropConnect (individual connections randomly zeroed):")
print(W_dc)
print("\nKey: Dropout removes NODES; DropConnect removes individual WEIGHTS.")

=== Dropout: Training Phase ===

Layer outputs BEFORE dropout (4 examples x 6 nodes):
[[ 0.4967 -0.1383  0.6477  1.523  -0.2342 -0.2341]
 [ 1.5792  0.7674 -0.4695  0.5426 -0.4634 -0.4657]
 [ 0.242  -1.9133 -1.7249 -0.5623 -1.0128  0.3142]
 [-0.908  -1.4123  1.4656 -0.2258  0.0675 -1.4247]]

Mini-batch 1: mask = [1 0 1 0 0 1]  dropped nodes: [1, 3, 4]
  Output (example 1): [ 0.4967 -0.      0.6477  0.     -0.     -0.2341]
  Active nodes: 3/6  (thinned network)

Mini-batch 2: mask = [1 1 0 0 1 0]  dropped nodes: [2, 3, 5]
  Output (example 1): [ 0.4967 -0.1383  0.      0.     -0.2342 -0.    ]
  Active nodes: 3/6  (thinned network)

Mini-batch 3: mask = [0 0 0 1 0 1]  dropped nodes: [0, 1, 2, 4]
  Output (example 1): [ 0.     -0.      0.      1.523  -0.     -0.2341]
  Active nodes: 2/6  (thinned network)

=== Test Phase: No Dropout, Scale Weights ===

At test time: all 6 nodes active BUT weights * p_keep=0.5
  Original weight row 1: [ 0.2089 -1.9597 -1.3282]
  Scaled   weight row 1: [ 0.1

## Example 2: Batch Normalization — Manual Calculation

We compute Batch Normalization step by step on a small hidden layer: 4 neurons, mini-batch of 5 examples. Step 1 computes the mean and variance per neuron across all 5 examples (column-wise). Step 2 subtracts the mean and divides by standard deviation — producing s_hat with mean=0, std=1 for every neuron. This is verified explicitly. Step 3 applies the learnable γ and β parameters (scale and shift), giving the final output z. The last section proves why BatchNorm is needed: two mini-batches with completely different raw distributions (means 2.04 vs 7.88) both become mean=0, std=1 after normalization — the internal covariate shift is eliminated.

In [2]:
import numpy as np

np.random.seed(42)
s = np.array([[2.1,-0.5,3.2,0.8],
              [1.8, 0.3,3.5,1.2],
              [2.5,-0.9,2.8,0.5],
              [1.5, 0.7,3.0,0.9],
              [2.3,-0.2,3.4,1.1]])  # 5 examples x 4 neurons

print("Hidden layer output s (5 examples x 4 neurons):")
print(s)

# Step 1: mean and variance across mini-batch (per neuron = column-wise)
mu    = s.mean(axis=0)
var   = s.var(axis=0)
sigma = np.sqrt(var + 1e-8)
print(f"\nStep 1: Per-neuron statistics (computed across all 5 examples):")
print(f"  mean mu   = {mu.round(4)}")
print(f"  var  s^2  = {var.round(4)}")
print(f"  std  sigma= {sigma.round(4)}")

# Step 2: normalize
s_hat = (s - mu) / sigma
print(f"\nStep 2: Normalized s_hat = (s - mu) / sigma:")
print(s_hat.round(4))
print(f"  Verify mean : {s_hat.mean(axis=0).round(6)}  <- all ~0")
print(f"  Verify std  : {s_hat.std(axis=0).round(6)}  <- all ~1")

# Step 3: scale and shift with learnable parameters
gamma = np.array([1.5, 0.8, 2.0, 1.2])
beta  = np.array([0.1, 0.0, -0.5, 0.3])
z = gamma * s_hat + beta
print(f"\nStep 3: z = gamma * s_hat + beta")
print(f"  gamma={gamma}  beta={beta}")
print(f"Final output z:\n{z.round(4)}")

print("\n=== Internal Covariate Shift Demo ===\n")
mb1 = np.array([2.1,1.8,2.5,1.5,2.3])  # mini-batch 1 outputs (neuron 1)
mb2 = np.array([8.4,7.2,9.0,6.0,8.8])  # mini-batch 2 outputs (same neuron, shifted)
bn1 = (mb1 - mb1.mean()) / mb1.std()
bn2 = (mb2 - mb2.mean()) / mb2.std()
print(f"Mini-batch 1 raw: mean={mb1.mean():.2f}, std={mb1.std():.2f}")
print(f"Mini-batch 2 raw: mean={mb2.mean():.2f}, std={mb2.std():.2f}")
print(f"After BN batch 1: mean={bn1.mean():.2f}, std={bn1.std():.2f}")
print(f"After BN batch 2: mean={bn2.mean():.2f}, std={bn2.std():.2f}")
print("\nBoth batches normalized to same distribution -> covariate shift eliminated!")

Hidden layer output s (5 examples x 4 neurons):
[[ 2.1 -0.5  3.2  0.8]
 [ 1.8  0.3  3.5  1.2]
 [ 2.5 -0.9  2.8  0.5]
 [ 1.5  0.7  3.   0.9]
 [ 2.3 -0.2  3.4  1.1]]

Step 1: Per-neuron statistics (computed across all 5 examples):
  mean mu   = [ 2.04 -0.12  3.18  0.9 ]
  var  s^2  = [0.1264 0.3216 0.0656 0.06  ]
  std  sigma= [0.3555 0.5671 0.2561 0.2449]

Step 2: Normalized s_hat = (s - mu) / sigma:
[[ 0.1688 -0.6701  0.0781 -0.4082]
 [-0.6751  0.7406  1.2494  1.2247]
 [ 1.2939 -1.3754 -1.4837 -1.633 ]
 [-1.5189  1.446  -0.7028  0.    ]
 [ 0.7313 -0.1411  0.859   0.8165]]
  Verify mean : [-0. -0. -0.  0.]  <- all ~0
  Verify std  : [1. 1. 1. 1.]  <- all ~1

Step 3: z = gamma * s_hat + beta
  gamma=[1.5 0.8 2.  1.2]  beta=[ 0.1  0.  -0.5  0.3]
Final output z:
[[ 0.3531 -0.5361 -0.3438 -0.1899]
 [-0.9126  0.5925  1.9988  1.7697]
 [ 2.0408 -1.1003 -3.4673 -1.6596]
 [-2.1783  1.1568 -1.9056  0.3   ]
 [ 1.197  -0.1129  1.2179  1.2798]]

=== Internal Covariate Shift Demo ===

Mini-batch 1 ra

## Example 3: Batch Normalization vs Layer Normalization — Side by Side

The final example clarifies the most commonly confused pair: Batch Normalization vs Layer Normalization. Both normalize to mean=0, std=1 — but they normalize along different axes. BatchNorm normalizes each neuron (column) across the mini-batch — so neuron 1's output across all 3 examples becomes standardized. LayerNorm normalizes each example (row) across its own neurons — so example 1's outputs across all 4 neurons become standardized. Look at the LayerNorm output: rows 1 and 2 come out identical, because row 2 is just row 1 shifted up by 4, and LayerNorm removes each example's own shift and scale. Row 3 has a different shape, so it stays different. The practical implication: BatchNorm needs a large batch to estimate good statistics. LayerNorm works with batch size 1. That's why Transformers use LayerNorm — they process sequences where batch size may be 1.

In [3]:
import numpy as np

H = np.array([[1., 2., 3., 4.],
              [5., 6., 7., 8.],
              [1., 2., 6., 7.]])  # 3 examples x 4 neurons

print("Hidden layer output H (3 examples x 4 neurons):")
print(H)

print("\n=== Batch Normalization ===")
print("Normalize each COLUMN (neuron) across all 3 examples\n")
mu_bn  = H.mean(axis=0)      # shape (4,)
std_bn = H.std(axis=0) + 1e-8
H_bn   = (H - mu_bn) / std_bn
print(f"Column means: {mu_bn.round(4)}")
print(f"Column stds : {std_bn.round(4)}")
print("BN output:")
print(H_bn.round(4))
print(f"Verify: mean per column = {H_bn.mean(axis=0).round(4)}  (all ~0)")
print(f"        std  per column = {H_bn.std(axis=0).round(4)}  (all ~1)")

print("\n=== Layer Normalization ===")
print("Normalize each ROW (example) across its own 4 neurons\n")
mu_ln  = H.mean(axis=1, keepdims=True)   # shape (3,1)
std_ln = H.std(axis=1, keepdims=True) + 1e-8
H_ln   = (H - mu_ln) / std_ln
print(f"Row means: {mu_ln.flatten().round(4)}")
print(f"Row stds : {std_ln.flatten().round(4)}")
print("LN output:")
print(H_ln.round(4))
print(f"Verify: mean per row = {H_ln.mean(axis=1).round(4)}  (all ~0)")
print(f"        std  per row = {H_ln.std(axis=1).round(4)}  (all ~1)")

print("\n=== Summary ===")
print("  Batch Norm : normalizes across BATCH (columns) — needs large batch")
print("  Layer Norm : normalizes across FEATURES (rows) — works with batch=1")
print("  Use BatchNorm for CNNs and MLPs; LayerNorm for Transformers/RNNs")

Hidden layer output H (3 examples x 4 neurons):
[[1. 2. 3. 4.]
 [5. 6. 7. 8.]
 [1. 2. 6. 7.]]

=== Batch Normalization ===
Normalize each COLUMN (neuron) across all 3 examples

Column means: [2.3333 3.3333 5.3333 6.3333]
Column stds : [1.8856 1.8856 1.6997 1.6997]
BN output:
[[-0.7071 -0.7071 -1.3728 -1.3728]
 [ 1.4142  1.4142  0.9806  0.9806]
 [-0.7071 -0.7071  0.3922  0.3922]]
Verify: mean per column = [-0. -0.  0.  0.]  (all ~0)
        std  per column = [1. 1. 1. 1.]  (all ~1)

=== Layer Normalization ===
Normalize each ROW (example) across its own 4 neurons

Row means: [2.5 6.5 4. ]
Row stds : [1.118  1.118  2.5495]
LN output:
[[-1.3416 -0.4472  0.4472  1.3416]
 [-1.3416 -0.4472  0.4472  1.3416]
 [-1.1767 -0.7845  0.7845  1.1767]]
Verify: mean per row = [ 0.  0. -0.]  (all ~0)
        std  per row = [1. 1. 1.]  (all ~1)

=== Summary ===
  Batch Norm : normalizes across BATCH (columns) — needs large batch
  Layer Norm : normalizes across FEATURES (rows) — works with batch=1
  Use B